In [ ]:
import os
import smtplib
import pandas as pd
from email.message import EmailMessage
from PIL import Image, ImageDraw, ImageFont

# ==============================================================================
# CONFIGURATION
# ==============================================================================
SMTP_SERVER = 'smtp.gmail.com'
SMTP_PORT = 465
SENDER_EMAIL = 'isaacoluwaseyiajao@gmail.com'
SENDER_PASSWORD = 'autnmnmzputozewe'  # 16-character Google App Password (NO spaces)

CONFERENCE_TITLE = "Data Analytics for Scientific Research & Innovation"
EVENT_DATE = "August 12, 2026"

# Ensure output directory exists
os.makedirs('certificates', exist_ok=True)


# ==============================================================================
# CERTIFICATE GENERATOR FUNCTION (LOGO + SIGNATURES INCLUDED)
# ==============================================================================
def draw_certificate(participant_name, conference_title, date_str, output_pdf_path):
    width, height = 2400, 1700
    bg_color = (252, 252, 254)
    navy_blue = (0, 32, 96)
    accent_gold = (212, 175, 55)
    text_dark = (40, 40, 40)
    
    img = Image.new('RGB', (width, height), color=bg_color)
    draw = ImageDraw.Draw(img)

    # 1. Outer Frame & Corner Accents
    draw.rectangle([50, 50, width - 50, height - 50], outline=navy_blue, width=12)
    draw.rectangle([70, 70, width - 70, height - 70], outline=accent_gold, width=4)

    draw.polygon([(50, 50), (220, 50), (50, 220)], fill=navy_blue)
    draw.polygon([(65, 65), (200, 65), (65, 200)], fill=accent_gold)
    
    draw.polygon([(width - 50, height - 50), (width - 220, height - 50), (width - 50, height - 220)], fill=navy_blue)
    draw.polygon([(width - 65, height - 65), (width - 200, height - 65), (width - 65, height - 200)], fill=accent_gold)

    # 2. Load Fonts
    try:
        font_header = ImageFont.truetype("georgiab.ttf", 46)
        font_cert = ImageFont.truetype("georgiab.ttf", 80)
        font_sub = ImageFont.truetype("arial.ttf", 34)
        font_name = ImageFont.truetype("georgiab.ttf", 75)
        font_body = ImageFont.truetype("arial.ttf", 38)
        font_conf = ImageFont.truetype("georgiab.ttf", 44)
        font_small = ImageFont.truetype("arial.ttf", 32)
    except IOError:
        font_header = font_cert = font_sub = font_name = font_body = font_conf = font_small = ImageFont.load_default()

    # 3. Paste School Logo (Top Center)
    logo_filename = None
    for name in ["fpa_logo.png", "fpa_logo.PNG", "fpa_logo.jpg", "fpa_logo.jpeg"]:
        if os.path.exists(name):
            logo_filename = name
            break

    if logo_filename:
        try:
            logo = Image.open(logo_filename).convert("RGBA")
            aspect_ratio = logo.width / logo.height
            new_height = 180
            new_width = int(new_height * aspect_ratio)
            logo = logo.resize((new_width, new_height))
            
            logo_x = int((width - new_width) / 2)
            logo_y = 90
            img.paste(logo, (logo_x, logo_y), mask=logo if logo.mode == 'RGBA' else None)
        except Exception as e:
            print(f"Warning: Could not process logo ({e})")

    # 4. Institution Header
    draw.text((width / 2, 300), "SCHOOL OF PURE AND APPLIED SCIENCES (SPAS)", fill=navy_blue, font=font_header, anchor="mm")
    draw.text((width / 2, 355), "FEDERAL POLYTECHNIC, ADO-EKITI", fill=text_dark, font=font_sub, anchor="mm")
    draw.line([(width / 2 - 300, 395), (width / 2 + 300, 395)], fill=accent_gold, width=3)

    # 5. Title & Recipient Name
    draw.text((width / 2, 480), "CERTIFICATE OF PARTICIPATION", fill=navy_blue, font=font_cert, anchor="mm")
    draw.text((width / 2, 570), "This is to certify that", fill=text_dark, font=font_sub, anchor="mm")
    draw.text((width / 2, 670), participant_name, fill=navy_blue, font=font_name, anchor="mm")
    draw.line([(width / 2 - 450, 720), (width / 2 + 450, 720)], fill=accent_gold, width=4)

    # 6. Event Description
    draw.text((width / 2, 800), "has actively participated in the workshop/conference on", fill=text_dark, font=font_body, anchor="mm")
    draw.text((width / 2, 880), f'"{conference_title}"', fill=navy_blue, font=font_conf, anchor="mm")
    draw.text((width / 2, 970), f"Held on {date_str}", fill=text_dark, font=font_body, anchor="mm")

    # 7. Gold Seal Badge
    badge_x, badge_y = width / 2, 1180
    badge_r = 80
    draw.ellipse([badge_x - badge_r, badge_y - badge_r, badge_x + badge_r, badge_y + badge_r], fill=accent_gold, outline=navy_blue, width=4)
    draw.ellipse([badge_x - badge_r + 10, badge_y - badge_r + 10, badge_x + badge_r - 10, badge_y + badge_r - 10], outline=(255, 255, 255), width=3)
    draw.text((badge_x, badge_y - 15), "SPAS", fill=navy_blue, font=font_sub, anchor="mm")
    draw.text((badge_x, badge_y + 25), "2026", fill=navy_blue, font=font_small, anchor="mm")

    # 8. Signatures Block
    # --- Left Signatory (Facilitator) ---
    draw.line([(350, 1450), (800, 1450)], fill=text_dark, width=2)
    draw.text((575, 1480), "Workshop Facilitator / Convener", fill=text_dark, font=font_small, anchor="mm")
    draw.text((575, 1520), "Dr. Isaac O. Ajao", fill=navy_blue, font=font_sub, anchor="mm")

    try:
        sig_fac = Image.open("sig_facilitator.png").convert("RGBA")
        sig_fac = sig_fac.resize((280, 90))
        img.paste(sig_fac, (435, 1350), mask=sig_fac)
    except IOError:
        pass  # Proceeds without crashing if image isn't present

    # --- Right Signatory (Committee Chairman) ---
    draw.line([(width - 800, 1450), (width - 350, 1450)], fill=text_dark, width=2)
    draw.text((width - 575, 1480), "Dean / Committee Chairman", fill=text_dark, font=font_small, anchor="mm")
    draw.text((width - 575, 1520), "School of Pure & Applied Sciences", fill=navy_blue, font=font_sub, anchor="mm")

    try:
        sig_chair = Image.open("sig_chairman.png").convert("RGBA")
        sig_chair = sig_chair.resize((280, 90))
        img.paste(sig_chair, (width - 715, 1350), mask=sig_chair)
    except IOError:
        pass  # Proceeds without crashing if image isn't present

    # 9. Save PDF Output
    img.save(output_pdf_path, "PDF", resolution=300.0)


# ==============================================================================
# BATCH EXECUTION & AUTOMATED MAILING LOOP
# ==============================================================================
# Read participants list
df = pd.read_csv('participants.csv')
print(f"Loaded {len(df)} participants from participants.csv\n")

# Connect to Gmail SMTP
with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT) as server:
    server.login(SENDER_EMAIL, SENDER_PASSWORD)
    print("Authentication successful! Generating and sending certificates...\n")

    for index, row in df.iterrows():
        name = row['Name']
        recipient_email = row['Email']

        clean_name = name.replace(' ', '_').replace('.', '')
        pdf_filename = f"certificates/Certificate_{clean_name}.pdf"

        # 1. Generate unique PDF certificate
        draw_certificate(
            participant_name=name,
            conference_title=CONFERENCE_TITLE,
            date_str=EVENT_DATE,
            output_pdf_path=pdf_filename
        )

        # 2. Construct Email Message
        msg = EmailMessage()
        msg['Subject'] = f'Certificate of Participation — {CONFERENCE_TITLE}'
        msg['From'] = SENDER_EMAIL
        msg['To'] = recipient_email
        msg.set_content(
            f"Dear {name},\n\n"
            f"Thank you for participating in the workshop/conference on '{CONFERENCE_TITLE}'.\n\n"
            f"Please find attached your official Certificate of Participation.\n\n"
            f"Best regards,\n"
            f"Dr. Isaac O. Ajao\n"
            f"Department of Statistics, Federal Polytechnic, Ado-Ekiti"
        )

        # 3. Attach PDF Certificate
        with open(pdf_filename, 'rb') as f:
            file_data = f.read()
            msg.add_attachment(file_data, maintype='application', subtype='pdf', filename=f"Certificate_{clean_name}.pdf")

        # 4. Send Email
        server.send_message(msg)
        print(f"✅ Generated & emailed certificate to: {name} ({recipient_email})")

print("\n🎉 All participant certificates processed and dispatched successfully!")